In [6]:
"""Install Ultralytics — Kaggle's base image ships PyTorch but not ultralytics."""
!pip install -q ultralytics

# NazarBaan — YOLOv8n Merged-Dataset Training (Phase 4)

**Project:** NazarBaan ANPR (Pakistani gated-community license plate detection)  
**This notebook:** Re-trains YOLOv8n on the merged dataset (Burhan Khan + ubaidp1049) using **identical hyperparameters** to the baseline. Any change in metrics is attributable to data, not training configuration.

## Comparison setup

| Aspect | Baseline (Phase 3) | This run (Phase 4) |
|---|---|---|
| Dataset | burhan-khan/pk-number-plates v1 | Merged: BK + ubaidp1049, deduplicated |
| Train images | 859 | 1003 |
| Valid images | 280 | 187 |
| Test images | 69 | 65 |
| Model | YOLOv8n | YOLOv8n |
| imgsz | 960 | 960 |
| epochs / patience | 100 / 20 | 100 / 20 |
| batch | 16 | 16 |
| seed | 42 | 42 |

## Hypothesis

Adding ubaidp1049 introduces close-up plates (2.87× larger median area) absent from Burhan Khan's distance-dominated distribution. Expected effects:

- Validation **recall should improve** (more plate-size variety = fewer misses on unusual sizes).
- Validation **precision may dip slightly** (more diverse plates = harder task) and that's acceptable.
- Headline **mAP@0.5 should hold or improve** if the merge is genuinely useful.

If mAP@0.5 *drops* substantially, the merge hurts — and that's still a finding worth reporting honestly.

In [8]:
"""Verify GPU is live and Ultralytics is the version we expect.
Kaggle preinstalls ultralytics, so we just import and check."""

import torch
import ultralytics
from ultralytics import YOLO

print(f"PyTorch:      {torch.__version__}")
print(f"Ultralytics:  {ultralytics.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"Device:       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM (GB):    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}" if torch.cuda.is_available() else "")

PyTorch:      2.10.0+cu128
Ultralytics:  8.4.52
CUDA:         True
Device:       Tesla T4
VRAM (GB):    15.6


## Step 1 — Patch the dataset config

The dataset was authored on Roboflow's filesystem, so its `data.yaml` has paths like `train: ../train/images` — relative to wherever the YAML lives. On Kaggle our data is mounted **read-only** at `/kaggle/input/nazarbaan-pk-plates-v1/`, so I can't edit that file in place.

The clean fix: **write a new `data.yaml`** under `/kaggle/working/` (writable) that points at the mounted data with absolute paths. YOLO will use this one and ignore the original.

In [9]:
"""Write a Kaggle-aware data.yaml that points at the merged dataset.
Same logic as the baseline notebook, only the path changes."""

from pathlib import Path
import yaml

DATA_ROOT = Path("/kaggle/input/datasets/hamzaasifff/nazarbaan-pk-plates-merged-v1")
WORK_DIR = Path("/kaggle/working")

with open(DATA_ROOT / "data.yaml") as f:
    src = yaml.safe_load(f)

patched = {
    "path": str(DATA_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": src["nc"],
    "names": src["names"],
}

patched_path = WORK_DIR / "data.yaml"
with open(patched_path, "w") as f:
    yaml.safe_dump(patched, f, sort_keys=False)

print(f"Patched data.yaml written to: {patched_path}")
print("---")
print(patched_path.read_text())

Patched data.yaml written to: /kaggle/working/data.yaml
---
path: /kaggle/input/datasets/hamzaasifff/nazarbaan-pk-plates-merged-v1
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- Number-Plate



## Step 2 — Fine-tune YOLOv8n

I loaded the COCO-pretrained `yolov8n.pt` (Ultralytics auto-downloads ~6 MB on first call) and fine-tune it on our patched dataset. The pretrained weights give me a head start — the backbone already knows what shapes and edges look like; I only need to teach it the concept of "plate."

**`project` and `name` parameters:** all training artifacts go to `runs/detect/baseline_yolov8n/` — weights, plots, confusion matrix, validation predictions. This folder is what I'll download at the end.

Training takes roughly **20–40 minutes** on a T4. The cell streams per-epoch metrics live (precision, recall, mAP50, mAP50-95). Watch the **val mAP50** trend — that's the headline number.

In [10]:
"""Fine-tune YOLOv8n on the merged dataset. Identical hyperparameters
to the baseline; only the `name` differs so artifacts go to a separate folder."""

model = YOLO("yolov8n.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=100,
    patience=20,
    imgsz=960,
    batch=16,
    project="/kaggle/working/runs/detect",
    name="merged_yolov8n",       # <-- changed
    pretrained=True,
    optimizer="auto",
    seed=42,
    deterministic=True,
    verbose=True,
    plots=True,
)

Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=merged_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

## Step 3 — Evaluate on the test set

The training loop only ever saw the *validation* split for model selection. The **test split** (69 images) is the genuinely held-out set — I touch it exactly once, here, to report the model's real-world generalization. Doing this gives us a number we can quote in the report with a straight face.

In [11]:
"""Run validation against the test split using the best checkpoint."""

best_weights = Path("/kaggle/working/runs/detect/merged_yolov8n/weights/best.pt")
print(f"Loading best weights: {best_weights}")
print(f"  File size: {best_weights.stat().st_size / 1e6:.1f} MB")

best_model = YOLO(str(best_weights))

test_metrics = best_model.val(
    data="/kaggle/working/data.yaml",
    split="test",
    imgsz=960,
    plots=True,
    project="/kaggle/working/runs/detect",
    name="merged_yolov8n_test",
)

print("\n=== Test set metrics ===")
print(f"mAP@0.5:      {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {test_metrics.box.map:.4f}")
print(f"Precision:    {test_metrics.box.mp:.4f}")
print(f"Recall:       {test_metrics.box.mr:.4f}")

Loading best weights: /kaggle/working/runs/detect/merged_yolov8n/weights/best.pt
  File size: 6.3 MB
Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 2.0±0.3 ms, read: 6.8±1.6 MB/s, size: 24.6 KB)
val: Scanning /kaggle/input/datasets/hamzaasifff/nazarbaan-pk-plates-merged-v1/test/labels... 65 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 65/65 283.6it/s 0.2s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/hamzaasifff/nazarbaan-pk-plates-merged-v1/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.0it/s 2.5s0.8s
                   all         65         68      0.985      0.953      0.993      0.768
Speed: 10.2ms preprocess, 8.9ms inference, 0.0ms loss, 5.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/m

## Step 4 — Package training artifacts

I bundle the trained weights, training curves, confusion matrix, and metrics into a single archive so I can download one file and commit it locally with a clean record of what this run produced.

In [12]:
"""Zip up everything we'll want offline: weights, plots, config, metrics."""

import shutil
import json

OUT_DIR = Path("/kaggle/working/merged_yolov8n_artifacts")
OUT_DIR.mkdir(exist_ok=True)

# Copy the training run folder (weights, plots, args.yaml, results.csv)
shutil.copytree(
    "/kaggle/working/runs/detect/merged_yolov8n",
    OUT_DIR / "train",
    dirs_exist_ok=True,
)
# Copy the test eval folder (test confusion matrix, PR curves on test)
shutil.copytree(
    "/kaggle/working/runs/detect/merged_yolov8n_test",
    OUT_DIR / "test",
    dirs_exist_ok=True,
)
# Copy the patched yaml
shutil.copy("/kaggle/working/data.yaml", OUT_DIR / "data.yaml")

# Drop a machine-readable summary
summary = {
    "model": "YOLOv8n",
    "dataset": "merged_v1 (burhan_khan + ubaidp1049, deduped, 1255 images)",
    "imgsz": 960,
    "epochs_configured": 100,
    "patience": 20,
    "batch": 16,
    "test_metrics": {
        "mAP50":      float(test_metrics.box.map50),
        "mAP50_95":   float(test_metrics.box.map),
        "precision":  float(test_metrics.box.mp),
        "recall":     float(test_metrics.box.mr),
    },
}
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Zip the whole bundle for one-click download
archive_path = shutil.make_archive(
    "/kaggle/working/merged_yolov8n_artifacts", "zip", OUT_DIR
)
print(f"\nArtifact bundle ready: {archive_path}")
print(f"Size: {Path(archive_path).stat().st_size / 1e6:.1f} MB")
print("\nDownload from Kaggle's right sidebar → Output → baseline_yolov8n_artifacts.zip")


Artifact bundle ready: /kaggle/working/merged_yolov8n_artifacts.zip
Size: 22.5 MB

Download from Kaggle's right sidebar → Output → baseline_yolov8n_artifacts.zip


In [13]:
"""Diagnostic — what files are sitting in /kaggle/working/ right now?"""

from pathlib import Path

work = Path("/kaggle/working")
print(f"Contents of {work}:")
for p in sorted(work.iterdir()):
    size = p.stat().st_size if p.is_file() else "-"
    print(f"  {p.name:<50} {size}")

# Also look for any zip
print("\nAll .zip files anywhere in /kaggle/working/:")
for z in work.rglob("*.zip"):
    print(f"  {z}  ({z.stat().st_size / 1e6:.1f} MB)")

# And confirm the merged_yolov8n folders exist
print("\nLooking for trained-model folders:")
for f in ["runs/detect/merged_yolov8n", "runs/detect/merged_yolov8n_test", "merged_yolov8n_artifacts"]:
    p = work / f
    print(f"  {f}: {'EXISTS' if p.exists() else 'MISSING'}")

Contents of /kaggle/working:
  .virtual_documents                                 -
  data.yaml                                          155
  merged_yolov8n_artifacts                           -
  merged_yolov8n_artifacts.zip                       22527731
  runs                                               -
  yolo26n.pt                                         5544453
  yolov8n.pt                                         6549796

All .zip files anywhere in /kaggle/working/:
  /kaggle/working/merged_yolov8n_artifacts.zip  (22.5 MB)

Looking for trained-model folders:
  runs/detect/merged_yolov8n: EXISTS
  runs/detect/merged_yolov8n_test: EXISTS
  merged_yolov8n_artifacts: EXISTS
